# Investigación: municipio 27493 (Nuevo Belén de Bajirá, Chocó) en los catálogos del profesor

Este cuaderno **no es parte del EDA principal** (no se referencia desde `catalogo_paneles.ipynb`
ni desde `GUIA_CODIGO.md`) -- es un cuaderno de investigación puntual para resolver el pendiente
abierto en la sección 6 de `catalogo_paneles.ipynb`: el único municipio de nuestro panel propio
(`cod_dane = 27493`, Nuevo Belén de Bajirá, Chocó) que no cruza contra `Municipios.CODIGO` del
profesor.

**Por qué existe este cuaderno aparte:** antes de decidir cómo tratar ese hueco (¿falta el dato,
o solo le faltan etiquetas?), hay que saber con certeza qué información existe realmente en cada
tabla del profesor para este municipio -- no basta con mirar una muestra de 5 filas sin orden
definido (`LIMIT 5` sin `ORDER BY`), que es lo único que se tenía hasta ahora.

**Estructura:**
1. Lo que ya se puede responder con los catálogos que ya están descargados localmente en
   `datos/profesor/` (`Municipios.CODIGO`, EVA, ICA pecuario) -- **no necesita BigQuery**.
2. Lo que falta (`FINAGRO_Desembolsos_EFECTIVA`, `SFC414_DASH`) -- esas dos tablas son demasiado
   grandes para descargarlas completas (6.16M y ~2M filas), así que para esta parte sí hace falta
   una consulta puntual a BigQuery. Usa la misma credencial de cuenta de servicio
   (`pipeline-satelital`) que ya usa `describir_fuentes_profesor.py` -- **no hace falta entrar a
   la consola web de BigQuery ni iniciar sesión con una cuenta personal**, solo que exista
   `datos/logs/bigquery-key.json` (el mismo archivo que ya usan los otros scripts del pipeline).

Ver `docs/GUIA_CODIGO.md` sección 10 y la nota de seguimiento de esta investigación para el
contexto completo de la decisión pendiente.

In [1]:
# Este notebook vive en notebooks/. Las rutas ("datos/...") y los imports
# del paquete pipeline/ se resuelven desde la raiz del repositorio, que es
# la carpeta que contiene main_local.py.
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / "main_local.py").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


## 1. Catálogos ya descargados localmente (sin BigQuery)

`Municipios.CODIGO`, la tabla de producción agropecuaria enriquecida (EVA) y el inventario
pecuario del ICA ya están cacheados completos en `datos/profesor/` (descargados antes con la API
gratuita `tabledata.list`, sin costo de consulta -- el mismo mecanismo que ya usa
`exploratorio_sfc414.ipynb` para SFC414). Como son tablas completas, no una muestra, lo que se
encuentre aquí es definitivo, no hay que correr nada en BigQuery para esta parte.

In [2]:
import pandas as pd
import unicodedata

CODIGO = "27493"

def quitar_tildes(s):
    """Normaliza nombres de municipio para buscar sin depender de tildes
    (ej. 'Belen' encuentra 'Belén'). Mismo criterio que la sección 6 de
    catalogo_paneles.ipynb usa para cod_dane: comparar por el dato, no por
    cómo esté escrito."""
    return "".join(c for c in unicodedata.normalize("NFD", str(s))
                    if unicodedata.category(c) != "Mn")

mun = pd.read_parquet("datos/profesor/municipios/parte-00000.parquet")
print(f"Municipios.CODIGO: {len(mun):,} filas cacheadas localmente")
print(f"  Municipios distintos en Chocó (departamento 27): "
      f"{mun.loc[mun['nombre_departamento'].str.contains('Choc', case=False, na=False), 'cod_dane'].nunique()}")
print()

print(f"Por código exacto (cod_dane == '{CODIGO}'):")
print(mun[mun["cod_dane"].astype(str) == CODIGO].to_string() or "  -- ninguna fila --")
print()

print("Por nombre (contiene 'bajir', sin tildes, para descartar que esté bajo otro código):")
coincide_nombre = mun["nombre_municipio"].apply(quitar_tildes).str.contains("bajir", case=False, na=False)
print(mun[coincide_nombre].to_string() or "  -- ninguna fila --")


Municipios.CODIGO: 20,196 filas cacheadas localmente
  Municipios distintos en Chocó (departamento 27): 30

Por código exacto (cod_dane == '27493'):
Empty DataFrame
Columns: [cod_dane, nombre_departamento, nombre_municipio, region, fecha, ruralidad]
Index: []

Por nombre (contiene 'bajir', sin tildes, para descartar que esté bajo otro código):
Empty DataFrame
Columns: [cod_dane, nombre_departamento, nombre_municipio, region, fecha, ruralidad]
Index: []


In [3]:
eva = pd.read_parquet("datos/profesor/eva/parte-00000.parquet")
print(f"EVA (producción agropecuaria): {len(eva):,} filas cacheadas localmente")

eva_27493 = eva[eva["Cod_mun"].astype(str) == CODIGO]
print(f"Filas con Cod_mun == '{CODIGO}': {len(eva_27493)}")
if len(eva_27493):
    print()
    print("Años cubiertos:", sorted(eva_27493["Year"].unique().tolist()))
    print("Cultivos reportados:", sorted(eva_27493["Cultivo"].dropna().unique().tolist()))
    print()
    print(eva_27493[["Year", "Cultivo", "Sembrada", "Cosechada", "Produccion", "Rendimiento", "Estado"]]
          .sort_values("Year").to_string(index=False))


EVA (producción agropecuaria): 347,141 filas cacheadas localmente
Filas con Cod_mun == '27493': 30

Años cubiertos: [datetime.date(2024, 1, 1)]
Cultivos reportados: ['Achiote', 'Aguacate', 'Ahuyama', 'Ají', 'Arroz', 'Borojó', 'Cacao', 'Caña', 'Coco', 'Frijol', 'Limón', 'Lulo', 'Maracuyá', 'Maíz', 'Palma de aceite', 'Pepino Cohombro', 'Piña', 'Plátano', 'Yuca', 'Zapote', 'Ñame']

      Year         Cultivo  Sembrada  Cosechada  Produccion  Rendimiento      Estado
2024-01-01         Achiote       5.0        5.0         5.0         1.00  Permanente
2024-01-01          Zapote      22.0       22.0        88.0         4.00  Permanente
2024-01-01            Yuca    2761.0        0.0         0.0         0.00 Transitorio
2024-01-01            Yuca    2500.0     2500.0     40000.0        16.00 Transitorio
2024-01-01         Plátano    2647.0     1720.0     44720.0        26.00  Permanente
2024-01-01            Piña      10.0       10.0        50.0         5.00  Permanente
2024-01-01 Pepino Cohom

In [4]:
ica = pd.read_parquet("datos/profesor/ica_pecuario/parte-00000.parquet")
print(f"ICA pecuario: {len(ica):,} filas cacheadas localmente")

ica_27493 = ica[ica["codigo_municipio"].astype(str) == CODIGO]
print(f"Filas con codigo_municipio == '{CODIGO}': {len(ica_27493)}")
if len(ica_27493):
    print(ica_27493[["anio", "municipio", "total_bovinos", "total_fincas_con_bovinos"]].to_string(index=False))
else:
    print("-- sin registros para este municipio en ICA pecuario --")


ICA pecuario: 8,976 filas cacheadas localmente
Filas con codigo_municipio == '27493': 0
-- sin registros para este municipio en ICA pecuario --


### Hallazgos de la sección 1 (confirmados contra las tablas completas, no una muestra)

Corrido el 11/10/2026 contra las copias locales completas de `datos/profesor/` -- esto no es una
muestra, es la tabla entera, así que es definitivo:

- **`Municipios.CODIGO`: ausencia confirmada, por código Y por nombre.** No hay ninguna fila con
  `cod_dane = 27493`, y tampoco ninguna fila cuyo `nombre_municipio` contenga "Bajir" bajo ningún
  departamento -- descarta la hipótesis de que el territorio esté ahí pero mal codificado. Chocó
  tiene **30 municipios** en este catálogo, no 31 (el número oficial). Esto coincide exactamente
  con lo que se vio en el tablero público del Observatorio (Looker Studio): también 30 municipios
  en Chocó, también sin resultados al buscar "Bajir"/"Belen" -- es el mismo catálogo
  desactualizado detrás de los dos.
- **EVA: el municipio SÍ tiene datos reales de producción agropecuaria -- 30 filas, todas de
  2024.** A diferencia de `Municipios.CODIGO`, la tabla de producción
  (`Produccion.eva_full_enriquecido`) sí tiene filas para `Cod_mun = 27493`: 21 cultivos distintos
  reportados (plátano, arroz, yuca, maíz, cacao, palma de aceite, entre otros), con área sembrada,
  cosechada, producción y rendimiento. Que **todas** las filas sean de 2024 tiene sentido con la
  causa raíz ya documentada (el municipio tomó posesión de su primer gobierno en junio de 2024) --
  es evidencia adicional a favor del diagnóstico, no un dato suelto.
- **ICA pecuario: cero filas para este municipio.** A diferencia de EVA, aquí no hay ningún
  registro -- puede ser porque el municipio genuinamente no reportó inventario pecuario todavía, o
  porque esta tabla depende de alguna otra referencia no investigada aquí.

**Conclusión de esta sección:** confirma y precisa lo que ya se sospechaba con la muestra de
FINAGRO -- `Municipios.CODIGO` es el catálogo que está desactualizado, no necesariamente las
tablas transaccionales/de producción en sí. El municipio existe y tiene datos reales en al menos
una de las dos tablas de producción (EVA, con 2024 completo), pero no en la otra (ICA pecuario) ni
en el catálogo de referencia. Falta confirmar si pasa lo mismo con FINAGRO y SFC414 (sección 2).

## 2. FINAGRO y SFC414 (necesita BigQuery)

Estas dos tablas son demasiado grandes para tener una copia local completa (6.16 millones y ~2
millones de filas), así que para saber si el municipio tiene créditos reales hace falta una
consulta puntual y agregada -- no una descarga completa.

**Credencial:** se reutiliza `_credenciales()` de `pipeline.entendimiento.describir_fuentes`, la
misma cuenta de servicio `pipeline-satelital` que ya usa `describir_fuentes_profesor.py` (que es
justo cómo se generó la muestra real de `docs/DESCRIPCION_FUENTES_PROFESOR.md` que ya mostró
créditos de este municipio). No hace falta `gcloud auth application-default login` ni iniciar
sesión con una cuenta personal de Google -- solo que exista `datos/logs/bigquery-key.json`.

In [5]:
from pipeline.entendimiento.describir_fuentes import _credenciales

credenciales = _credenciales()
if credenciales is None:
    print("No se encontro datos/logs/bigquery-key.json -- ver GUIA_CODIGO.md seccion 5.16.")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="ofr-credito-deforestacion", credentials=credenciales)
    print("Credencial de servicio cargada correctamente.")


C:\Users\busta\trabajo_de_grado_ofr\.venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Credencial de servicio cargada correctamente.


C:\Users\busta\trabajo_de_grado_ofr\.venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.cloud.bigquery once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery past that date.
  warnings.warn(message, FutureWarning)


In [6]:
# FINAGRO: cuenta, rango de fechas y monto total para este municipio -- y, por separado,
# si las columnas de enriquecimiento geografico (DPNOM/MPIO/DPMP/region/ruralidad/PDET) vienen
# nulas para TODAS las filas de este municipio o solo en la muestra de 5 que se vio antes.
consulta_finagro = f"""
SELECT
  COUNT(*) AS n_creditos,
  MIN(fecha_credito) AS fecha_min,
  MAX(fecha_credito) AS fecha_max,
  SUM(valor_credito) AS valor_total,
  COUNTIF(DPNOM IS NULL) AS n_dpnom_nulo,
  COUNTIF(MPIO IS NULL) AS n_mpio_nulo,
  COUNTIF(region IS NULL) AS n_region_nulo,
  COUNTIF(ruralidad IS NULL) AS n_ruralidad_nulo,
  COUNTIF(PDET IS NULL) AS n_pdet_nulo
FROM `prueba-ofr.Inclusion_Financiera.FINAGRO_Desembolsos_EFECTIVA`
WHERE cod_municipio = '{CODIGO}'
"""
resultado_finagro = client.query(consulta_finagro).result().to_dataframe()
print(resultado_finagro.to_string(index=False))


C:\Users\busta\trabajo_de_grado_ofr\.venv\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 n_creditos  fecha_min  fecha_max  valor_total  n_dpnom_nulo  n_mpio_nulo  n_region_nulo  n_ruralidad_nulo  n_pdet_nulo
        197 2024-07-19 2026-06-30 5298393300.0           197          197            197               197          197


In [7]:
# SFC414: lo mismo, aunque la muestra de 5 filas del reporte automatico no mostro
# ningun caso de este municipio -- con ~2 millones de filas eso no prueba nada en
# ningun sentido, por eso hace falta esta consulta agregada.
consulta_sfc414 = f"""
SELECT
  COUNT(*) AS n_registros,
  SUM(Montos_desembolsados) AS monto_total,
  SUM(Numero_de_creditos_desembolsados) AS creditos_total,
  MIN(Fecha_Corte) AS fecha_min,
  MAX(Fecha_Corte) AS fecha_max
FROM `prueba-ofr.Inclusion_Financiera.SFC414_DASH`
WHERE Codigo_Municipio = '{CODIGO}'
"""
resultado_sfc414 = client.query(consulta_sfc414).result().to_dataframe()
print(resultado_sfc414.to_string(index=False))


C:\Users\busta\trabajo_de_grado_ofr\.venv\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 n_registros  monto_total  creditos_total  fecha_min  fecha_max
         202 6717495922.0             348 2024-09-27 2026-09-25


### Hallazgos de la sección 2

Corrido el 11/10/2026 por Luisa contra BigQuery en vivo (credencial de servicio
`pipeline-satelital`), consulta agregada sobre la tabla completa -- no una muestra.

**FINAGRO** (`prueba-ofr.Inclusion_Financiera.FINAGRO_Desembolsos_EFECTIVA`):

- 197 créditos desembolsados con `cod_municipio = '27493'`, entre 2024-07-19 y 2026-06-30.
- Monto total: $5.298.393.300 COP.
- El 100% de esos 197 créditos (197/197) tiene nulas las columnas de enriquecimiento
  geográfico: `DPNOM`, `MPIO`, `region`, `ruralidad`, `PDET`. No es un hueco parcial --
  es total, en toda la población, no solo en la muestra de 5 filas vista originalmente.
- Nota: esta misma consulta devolvió 107 créditos en una corrida anterior (misma mañana) y
  197 en esta corrida. FINAGRO es una tabla que el profesor actualiza de forma continua, así
  que la diferencia son créditos nuevos cargados entre una corrida y otra -- no un error de
  la consulta ni del pipeline. Cualquier cifra de este cuaderno debe leerse como "al momento
  de la corrida", no como un conteo fijo.

**SFC414** (`prueba-ofr.Inclusion_Financiera.SFC414_DASH`):

- 202 registros con `Codigo_Municipio = '27493'`, entre 2024-09-27 y 2026-09-25.
- Esos 202 registros agregan 348 créditos (`Numero_de_creditos_desembolsados`).
- Monto total: $6.717.495.922 COP.
- La muestra automática de 5 filas que generó `describir_fuentes_profesor.py` no incluía
  ningún caso de este municipio (solo mostraba `05001`, Medellín) -- con ~2 millones de
  filas en la tabla eso no probaba nada en ningún sentido, y en efecto sí hay actividad real.

**Lectura conjunta con la sección 1:** el municipio 27493 tiene actividad real y no trivial
en tres de las cuatro fuentes del profesor -- EVA (30 filas de producción agropecuaria 2024),
FINAGRO ($5.298 millones COP en 197 créditos) y SFC414 ($6.717 millones COP, 348 créditos) --
y cero presencia en `Municipios.CODIGO` y en ICA pecuario. El dinero y la producción existen;
lo que no existe es el municipio en el catálogo de referencia que debería etiquetarlo con
departamento, región, ruralidad y PDET. Esto pesa en contra de cualquier alternativa que
descarte o colapse la identidad propia del municipio (por ejemplo, tratarlo únicamente como
Chocó-departamento): se perdería la trazabilidad de créditos y producción reales y verificables,
no solo una etiqueta demográfica.

## Siguiente paso

Con los resultados de las dos secciones ya completos, volver a la nota de seguimiento de esta
investigación (`municipio-27493-belen-de-bajira.md`, en el proyecto de Claude) para retomar la
decisión de tratamiento con la evidencia completa. Sigue sin implementarse nada en
`catalogo_paneles.ipynb` ni en `docs/GUIA_CODIGO.md` -- eso espera a que Luisa confirme la
alternativa elegida.
